# Image and System Analysis | Division of Medical Radiation Physics | Stockholm University
```mehdi.astaraki@fysik.su.se```

# Python Programming Introduction - Part 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Astarakee/isa-su/blob/main/python_intro/pt_intro3.ipynb)

Welcome to Part 3! In this notebook, we move beyond the fundamentals and arrays to explore the broader Python data ecosystem. We will cover how to visualize data, process images, use heavy-lifting data science libraries, and save our work.

We will cover:
1. Data Visualization (Matplotlib & Seaborn)
2. Computer Vision with OpenCV (cv2)
3. The Heavy Lifters (Pandas, SciPy, & Scikit-Learn)
4. Data Serialization & Saving Your Work

Let's dive in!


---
## SECTION 0: Google Colab Environment Setup & Asset Check

### Theoretical & Engineering Setup
To ensure seamless cross-platform execution (whether locally or inside Google Colab), this environment setup cell automatically downloads the required image asset (`Lenna.png`) if it is missing from the working directory.


In [ ]:
# Section 0: Google Colab Environment Setup & Asset Check
import os
import urllib.request

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/Astarakee/isa-su/main/python_intro"
ASSETS = {
    "Lenna.png": f"{GITHUB_RAW_BASE}/Lenna.png"
}

for fname, url in ASSETS.items():
    if not os.path.exists(fname):
        print(f"Downloading missing asset '{fname}'...")
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as resp, open(fname, 'wb') as f:
                f.write(resp.read())
            print(f"Successfully downloaded '{fname}'")
        except Exception as e:
            print(f"Warning: Failed to download '{fname}': {e}")

print("Environment setup complete. All required assets are ready.")

# Module 1: Data Visualization (Matplotlib & Seaborn)

This module teaches how to translate NumPy arrays into meaningful insights, covering both raw mathematical plotting and statistical analysis.

## 1. Mechanics of Plotting

### The Anatomy of a Figure
In Matplotlib, there is a strict hierarchy:
*   **Figure (`plt.figure()`)**: The overall blank canvas/window.
*   **Axes (`plt.axes()` or `ax`)**: The actual plot (the bounding box with ticks, labels, and the plotted lines). A single Figure can contain multiple Axes (subplots).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set a nice visual style
sns.set_theme(style="whitegrid")

# Create a Figure (canvas) and an Axes (plot)
fig, ax = plt.subplots(figsize=(8, 4))

ax.set_title("The Empty Canvas (Axes) inside a Figure")
ax.set_xlabel("X-axis")
ax.set_ylabel("Y-axis")
plt.show()


### Continuous vs. Discrete Plots
*   **Continuous**: Line plots (`plt.plot()`) are used for mathematical functions or continuous time-series data.
*   **Discrete**: Scatter plots (`plt.scatter()`) are used for independent, unconnected data points.


In [ ]:
# 1. Continuous (Line Plot)
x = np.linspace(0, 10, 100)
y_sin = np.sin(x)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, y_sin, color='blue', label='sin(x)')
ax1.set_title("Continuous: Line Plot")
ax1.legend()

# 2. Discrete (Scatter Plot)
# Let's add some noise to make it look like real independent measurements
y_noisy = y_sin + np.random.normal(0, 0.2, 100)

ax2.scatter(x, y_noisy, color='red', alpha=0.6, label='Noisy Data')
ax2.set_title("Discrete: Scatter Plot")
ax2.legend()

plt.tight_layout()
plt.show()


### Plotting Vectors
We use `plt.quiver()` to visualize directional vectors and forces (e.g., wind direction, magnetic fields).


In [ ]:
# Create a grid of points
X, Y = np.meshgrid(np.arange(-2, 3, 1), np.arange(-2, 3, 1))

# Define vector directions (e.g., a simple outward radial field)
U = X 
V = Y 

fig, ax = plt.subplots(figsize=(5, 5))
# Quiver plots the arrows. X, Y are positions; U, V are directions
ax.quiver(X, Y, U, V, color='purple')
ax.set_title("Vector Field using plt.quiver()")
ax.grid(True)
plt.show()


## 2. Statistical Distributions & Data Analysis Plots

We often need to understand the underlying *shape* of our data using Histograms and Probability Density Functions (PDFs).

### Plotting Famous Distributions


In [ ]:
# Generate synthetic datasets using NumPy
rng = np.random.default_rng(seed=42)

# 1. Gaussian (Normal): The classic bell curve
data_gaussian = rng.normal(loc=0, scale=1, size=1000)

# 2. Rayleigh: Common in signal processing and wind speeds
data_rayleigh = rng.rayleigh(scale=1, size=1000)

# 3. Power Law (Pareto): Models wealth distribution, file sizes, network traffic
data_power = rng.power(a=0.5, size=1000)

# Plotting them side-by-side using Seaborn's histplot (which overlays a density curve KDE)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(data_gaussian, kde=True, ax=axes[0], color='blue')
axes[0].set_title("Gaussian (Normal)")

sns.histplot(data_rayleigh, kde=True, ax=axes[1], color='green')
axes[1].set_title("Rayleigh")

sns.histplot(data_power, kde=True, ax=axes[2], color='red', bins=30)
axes[2].set_title("Power Law (Pareto)")

plt.tight_layout()
plt.show()


### Categorical & Summary Plots


In [ ]:
# Bar Plots: For discrete category comparisons
categories = ['Group A', 'Group B', 'Group C']
values = [15, 42, 28]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(categories, values, color=['#ff9999','#66b3ff','#99ff99'])
ax1.set_title("Bar Plot (Categorical)")

# Box Plots: Crucial for data cleaning. 
# Shows Median, Quartiles, and Outliers (the dots outside the whiskers).
box_data = [
    rng.normal(10, 2, 100),  # Well behaved
    rng.normal(15, 5, 100),  # Higher variance, more outliers
    rng.normal(5, 1, 100)    # Tightly packed
]

ax2.boxplot(box_data, tick_labels=['Set 1', 'Set 2', 'Set 3'])
ax2.set_title("Box Plot (Summary & Outliers)")

plt.show()


# Module 2: Computer Vision with OpenCV (cv2)

OpenCV is the industry standard for image and video processing. 
**The key realization:** Images are just multi-dimensional NumPy arrays!
*   A Grayscale image is a 2D matrix (Height x Width) where values represent brightness (0-255).
*   A Color image is a 3D Tensor (Height x Width x Channels).

### 1. `cv2.imshow` and the BGR vs. RGB Quirk
While most of the digital world (including Matplotlib) uses **RGB** (Red, Green, Blue) format, OpenCV historically reads and saves images in **BGR** (Blue, Green, Red) format. 
*If you plot an OpenCV image directly in Matplotlib without converting it, the reds and blues will be swapped (making people look like Smurfs)!*

Normally, when running scripts directly on your computer, you can use `cv2.imshow("Window Name", image)` to open an interactive GUI window. However, inside Jupyter Notebooks, `cv2.imshow` crashes the kernel. Instead, we must use `matplotlib.pyplot.imshow` (and convert the image to RGB first!).

Let's load a sample image from the local directory to work with.


In [ ]:
import cv2

# Load the local image "Lenna.png"
# Ensure the image file is in the same directory as the notebook
try:
    base_img = cv2.imread("Lenna.png")
    
    if base_img is None:
        raise FileNotFoundError("Image 'Lenna.png' could not be found or loaded.")
        
    # IMPORTANT: Convert BGR to RGB for correct Matplotlib display
    base_img_rgb = cv2.cvtColor(base_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(4, 4))
    plt.imshow(base_img_rgb)
    plt.title("1. Original Image (RGB)")
    plt.axis('off')
    plt.show()
    
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please make sure 'Lenna.png' is placed in the same folder as this notebook.")
    # For the sake of the tutorial running, we will create a synthetic image fallback
    print("Falling back to a synthetic image...")
    base_img = np.full((300, 300, 3), 255, dtype=np.uint8)
    cv2.circle(base_img, (150, 150), 50, (255, 0, 0), -1)
    base_img_rgb = cv2.cvtColor(base_img, cv2.COLOR_BGR2RGB)


### 2. Adding Noise (Gaussian & Salt and Pepper)
Real-world images are rarely perfect. Algorithms need to handle noise from faulty sensors, compression, or transmission errors.

*   **Gaussian Noise**: Simulates electronic sensor noise. It is random variation modeled by a normal (Gaussian) distribution.
*   **Salt & Pepper Noise**: Simulates transmission errors where random pixels become completely white (salt) or completely black (pepper).


In [ ]:
def add_gaussian_noise(image, mean=0, std=25):
    # Generate Gaussian noise of the same shape
    gauss = np.random.normal(mean, std, image.shape)
    # Add it to the image, and clip values to stay between 0 and 255
    noisy = np.clip(image + gauss, 0, 255).astype(np.uint8)
    return noisy

def add_salt_and_pepper_noise(image, amount=0.05):
    noisy = np.copy(image)
    # Generate random probabilities
    probs = np.random.random(noisy.shape[:2]) # 2D array of probabilities
    
    # Apply Salt (White) where probability < amount/2
    noisy[probs < (amount / 2)] = 255
    # Apply Pepper (Black) where probability > 1 - amount/2
    noisy[probs > 1 - (amount / 2)] = 0
    return noisy

img_gauss_noise = add_gaussian_noise(base_img_rgb)
img_sp_noise = add_salt_and_pepper_noise(base_img_rgb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(img_gauss_noise); ax1.set_title("2. Gaussian Noise"); ax1.axis('off')
ax2.imshow(img_sp_noise); ax2.set_title("2. Salt & Pepper Noise"); ax2.axis('off')
plt.show()


### 3. Filtering (Low Pass, Median, High Pass, Sobel)
Filters (or Kernels) slide across the image matrix, altering pixels based on their neighbors.

*   **Low Pass Filter (Blurring)**: Removes high-frequency details (noise) but blurs edges.
*   **Median Filter**: Excellent at removing Salt & Pepper noise because it replaces the pixel with the median of its neighbors, ignoring extreme outliers.
*   **High Pass Filter / Edge Detection**: Emphasizes sharp changes in intensity (edges) while ignoring smooth areas.
*   **Sobel Filters**: Specific directional edge detectors (calculating gradients in X or Y directions).


In [ ]:
# 1. Low Pass Filter (Gaussian Blur)
# Using a 5x5 kernel. Larger kernel = more blur.
blur_img = cv2.GaussianBlur(img_gauss_noise, (5, 5), 0)

# 2. Median Filter (Perfect for Salt & Pepper noise!)
median_img = cv2.medianBlur(img_sp_noise, 5)

# 3. High Pass / Edge Detection (Canny)
# We usually apply Canny to grayscale images.
gray_base = cv2.cvtColor(base_img, cv2.COLOR_BGR2GRAY)
canny_edges = cv2.Canny(gray_base, threshold1=100, threshold2=200)

# 4. Sobel Filters (Directional Gradients)
# Sobel X (Vertical Edges)
sobel_x = cv2.Sobel(gray_base, cv2.CV_64F, 1, 0, ksize=3)
# Sobel Y (Horizontal Edges)
sobel_y = cv2.Sobel(gray_base, cv2.CV_64F, 0, 1, ksize=3)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0,0].imshow(img_gauss_noise); axes[0,0].set_title("Noisy Input (Gaussian)"); axes[0,0].axis('off')
axes[0,1].imshow(blur_img); axes[0,1].set_title("Low Pass (Gaussian Blur)"); axes[0,1].axis('off')
axes[0,2].imshow(median_img); axes[0,2].set_title("Median Filter on S&P Noise"); axes[0,2].axis('off')

axes[1,0].imshow(canny_edges, cmap='gray'); axes[1,0].set_title("Canny Edges"); axes[1,0].axis('off')
axes[1,1].imshow(np.abs(sobel_x), cmap='gray'); axes[1,1].set_title("Sobel X (Vertical Edges)"); axes[1,1].axis('off')
axes[1,2].imshow(np.abs(sobel_y), cmap='gray'); axes[1,2].set_title("Sobel Y (Horizontal Edges)"); axes[1,2].axis('off')

plt.tight_layout()
plt.show()


### 4. Geometric Transformations (Rotation, Scaling, Shifting)
Modifying the spatial arrangement of the image pixels.

*   **Scaling (Resizing)**: Changing the resolution.
*   **Rotation**: Requires an "Affine Transformation Matrix" which OpenCV calculates for us via `cv2.getRotationMatrix2D`.
*   **Shifting (Translation)**: Sliding the image on the X or Y axis. We define a translation matrix: `M = [[1, 0, tx], [0, 1, ty]]`.


In [ ]:
height, width = base_img_rgb.shape[:2]

# 1. Scaling (Resize to half width, double height)
scaled_img = cv2.resize(base_img_rgb, (width//2, height*2))

# 2. Rotation (Rotate 45 degrees around the center)
center = (width // 2, height // 2)
# Matrix: center, angle, scale
rotation_matrix = cv2.getRotationMatrix2D(center, 45, 1.0)
# cv2.warpAffine applies the transformation matrix
rotated_img = cv2.warpAffine(base_img_rgb, rotation_matrix, (width, height))

# 3. Shifting (Translate Right by 100px, Down by 50px)
# Matrix: [1, 0, tx], [0, 1, ty]
translation_matrix = np.float32([[1, 0, 100], [0, 1, 50]])
shifted_img = cv2.warpAffine(base_img_rgb, translation_matrix, (width, height))


fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(scaled_img); axes[0].set_title("Scaling"); axes[0].axis('off')
axes[1].imshow(rotated_img); axes[1].set_title("Rotation (45 deg)"); axes[1].axis('off')
axes[2].imshow(shifted_img); axes[2].set_title("Shifting (Translation)"); axes[2].axis('off')
plt.show()


### 5. Thresholding (Simple, Adaptive, and Otsu's)
Thresholding converts a grayscale image into a binary image (pure black and white). It is heavily used in document scanning and separating foreground objects from backgrounds.

*   **Simple Thresholding**: Pick a hardcoded number (e.g., 127). Everything above it goes white, everything below goes black. Problem: Fails if lighting is uneven.
*   **Adaptive Thresholding**: The threshold value is calculated dynamically for smaller regions of the image. Great for uneven lighting (like shadows on a document).
*   **Otsu's Thresholding**: An algorithm that mathematically calculates the *perfect* global threshold value by analyzing the image's histogram distribution.


In [ ]:
# Let's create an image with uneven lighting (a gradient background)
gradient = np.linspace(0, 255, width).reshape(1, width)
gradient = np.repeat(gradient, height, axis=0).astype(np.uint8)

# Add some text (foreground objects)
uneven_img = gradient.copy()
cv2.putText(uneven_img, "HELLO WORLD", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 2, 0, 5)
cv2.putText(uneven_img, "TEST", (350, 400), cv2.FONT_HERSHEY_SIMPLEX, 2, 0, 5)


# 1. Simple Global Thresholding (Threshold = 127)
# Note: cv2.threshold returns a tuple (threshold_used, thresholded_image)
_, simple_thresh = cv2.threshold(uneven_img, 127, 255, cv2.THRESH_BINARY)

# 2. Adaptive Thresholding (Calculates local thresholds in 11x11 blocks)
adaptive_thresh = cv2.adaptiveThreshold(uneven_img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
                                        cv2.THRESH_BINARY, 11, 2)

# 3. Otsu's Thresholding (Calculates the optimal global threshold automatically)
# We pass 0 as the threshold because Otsu calculates it for us.
otsu_val, otsu_thresh = cv2.threshold(uneven_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0,0].imshow(uneven_img, cmap='gray'); axes[0,0].set_title("Original (Uneven Lighting)"); axes[0,0].axis('off')

# Simple fails because the right side of the gradient is brighter than the text!
axes[0,1].imshow(simple_thresh, cmap='gray'); axes[0,1].set_title("Simple Global (Thresh=127)"); axes[0,1].axis('off')

# Adaptive succeeds everywhere by looking at local contrast!
axes[1,0].imshow(adaptive_thresh, cmap='gray'); axes[1,0].set_title("Adaptive Thresholding"); axes[1,0].axis('off')

# Otsu calculates the optimal global value (which ended up being 127, so it looks like Simple in this specific synthetic case)
axes[1,1].imshow(otsu_thresh, cmap='gray'); axes[1,1].set_title(f"Otsu's Thresholding (Calculated Thresh={otsu_val})"); axes[1,1].axis('off')

plt.tight_layout()
plt.show()


### 6. Morphological Operators (Erosion & Dilation)
These are operations based on the image shape, normally performed on binary images (like the output of a threshold). They require a structuring element (a kernel).

*   **Erosion**: Erodes away the boundaries of foreground objects. (Useful for removing small white noise).
*   **Dilation**: Increases the white region in the image. (Useful for joining broken parts of an object).


In [ ]:
# Let's create a binary image with noise
binary_img = np.zeros((200, 200), dtype=np.uint8)
cv2.rectangle(binary_img, (50, 50), (150, 150), 255, -1) # Solid white square
# Add white noise outside
binary_img[20:30, 20:30] = 255
binary_img[170:180, 170:180] = 255
# Add black holes inside
binary_img[80:90, 80:90] = 0
binary_img[120:130, 120:130] = 0

# Define a 5x5 rectangular kernel
kernel = np.ones((5,5), np.uint8)

# 1. Erosion (Eats away the white regions, removes white noise, but expands black holes)
erosion = cv2.erode(binary_img, kernel, iterations=1)

# 2. Dilation (Expands the white regions, fills black holes, but expands white noise)
dilation = cv2.dilate(binary_img, kernel, iterations=1)

# 3. Opening (Erosion followed by Dilation). Perfect for removing white noise outside the object!
opening = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)

# 4. Closing (Dilation followed by Erosion). Perfect for filling black holes inside the object!
closing = cv2.morphologyEx(binary_img, cv2.MORPH_CLOSE, kernel)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(binary_img, cmap='gray'); axes[0].set_title("Original Binary w/ Noise"); axes[0].axis('off')
axes[1].imshow(erosion, cmap='gray'); axes[1].set_title("Erosion"); axes[1].axis('off')
axes[2].imshow(dilation, cmap='gray'); axes[2].set_title("Dilation"); axes[2].axis('off')
axes[3].imshow(opening, cmap='gray'); axes[3].set_title("Opening (Removes outer noise)"); axes[3].axis('off')
axes[4].imshow(closing, cmap='gray'); axes[4].set_title("Closing (Fills inner holes)"); axes[4].axis('off')

plt.show()


# Module 3: The Heavy Lifters (Pandas, SciPy, & Scikit-Learn)

Here is a high-level "taste test" of the conventional libraries that dominate the Python data ecosystem.

## 1. Pandas: The Data Wrangler
**Official Site:** [pandas.pydata.org](https://pandas.pydata.org/)

**What it is:** Pandas provides the **DataFrame**: think of it as the Python equivalent of a SQL table or an incredibly powerful, programmable Excel spreadsheet. It is built on top of NumPy but allows for labeled columns, text data, and mixed data types.
**Advantages:**
*   **Missing Data**: Incredibly robust tools for handling `NaN` and missing data (`.dropna()`, `.fillna()`).
*   **I/O**: Native support for reading/writing CSV, Excel, SQL, JSON, and Parquet files in one line of code.
*   **Time Series**: Unmatched capabilities for resampling and aligning dates and times.

**Real-World Application:** 
A retail company has a 10GB CSV file of customer purchases. They use Pandas to load the data, filter out incomplete transactions, group the data by customer demographics, and calculate the monthly average spending.


In [ ]:
import pandas as pd
import numpy as np

# --- Practical Real-World Example: Sales Data Wrangling ---

# 1. Simulate a messy real-world dataset
data = {
    'Date': pd.date_range(start='2023-01-01', periods=6, freq='D'),
    'Store_ID': ['S1', 'S2', 'S1', 'S3', 'S2', 'S1'],
    'Revenue': [1500, np.nan, 2100, 800, 3200, 1900], # Notice the missing data (NaN)
    'Employees_Working': [5, 3, 6, 2, 8, 5]
}

df = pd.DataFrame(data)
print("--- 1. Raw Loaded Data (Notice the NaN) ---")
print(df)

# 2. Data Cleaning: Fill missing revenue with the median revenue of ALL stores
median_rev = df['Revenue'].median()
df['Revenue'] = df['Revenue'].fillna(median_rev)

print("\n--- 2. Cleaned Data (NaN replaced with median) ---")
print(df)

# 3. Feature Engineering: Create a new column (Revenue per Employee)
df['Rev_Per_Employee'] = df['Revenue'] / df['Employees_Working']

# 4. Aggregation: Calculate total revenue and average Rev_Per_Employee per store
print("\n--- 3. Aggregated Store Performance ---")
store_performance = df.groupby('Store_ID').agg({
    'Revenue': 'sum',
    'Rev_Per_Employee': 'mean'
}).round(2)

print(store_performance)

# 5. I/O: Save the cleaned, aggregated report to a new CSV file
store_performance.to_csv("store_report.csv")
print("\n[Report saved to store_report.csv]")


## 2. SciPy: The Scientific Engine
**Official Site:** [scipy.org](https://scipy.org/)

**What it is:** SciPy builds on top of NumPy to provide a vast library of mathematical algorithms and convenience functions built for scientists and engineers.
**Advantages:**
*   Provides robust tools for **Optimization**, **Integration**, **Interpolation**, **Eigenvalue Problems**, and **Signal/Image Processing**.
*   It implements battle-tested algorithms written in Fortran and C, meaning it is mathematically rigorous and blazingly fast.

**Real-World Application:** 
An aerospace engineer needs to design the shape of an airplane wing to minimize air resistance (drag). They use `scipy.optimize` to tweak the parameters of the wing shape until the calculated drag reaches its absolute minimum.


In [ ]:
from scipy.optimize import minimize
import numpy as np 
import matplotlib.pyplot as plt

# --- Practical Real-World Example: Manufacturing Optimization ---
# Scenario: A factory makes cylindrical cans. They need a can that holds exactly 
# 330 ml (Volume = 330). They want to MINIMIZE the amount of aluminum metal used
# to save money. We need to find the optimal Radius (r) and Height (h).
# Surface Area (Cost) = 2 * pi * r^2 + 2 * pi * r * h
# Volume constraint: pi * r^2 * h = 330 => h = 330 / (pi * r^2)

# 1. Define the Objective Function (What we want to minimize: Surface Area)
def surface_area(r):
    # We substitute h using the volume constraint to make it a 1-variable problem
    h = 330 / (np.pi * r**2)
    area = 2 * np.pi * r**2 + 2 * np.pi * r * h
    return area

# 2. Run the Optimizer (Starting with a random guess of r = 1.0 cm)
# SciPy will iteratively test different radii until it finds the absolute minimum area.
result = minimize(surface_area, x0=np.array([1.0]))

optimal_radius = result.x[0]
optimal_height = 330 / (np.pi * optimal_radius**2)
min_area = result.fun

print("--- Optimization Results ---")
print(f"Optimal Radius: {optimal_radius:.2f} cm")
print(f"Optimal Height: {optimal_height:.2f} cm")
print(f"Minimum Aluminum Required: {min_area:.2f} cm²")

# 3. Let's visualize the curve to prove SciPy found the bottom!
radii = np.linspace(1, 10, 100)
areas = surface_area(radii)

plt.figure(figsize=(6, 3))
plt.plot(radii, areas, label='Surface Area Cost')
plt.scatter(optimal_radius, min_area, color='red', zorder=5, label='SciPy Optimal Point')
plt.title("Minimizing Aluminum Usage for a 330ml Can")
plt.xlabel("Radius (cm)")
plt.ylabel("Surface Area (cm²)")
plt.legend()
plt.show()


## 3. Scikit-Learn: The Machine Learning Standard
**Official Site:** [scikit-learn.org](https://scikit-learn.org/)

**What it is:** Scikit-Learn (sklearn) is the premier library for classical Machine Learning in Python. 
**Advantages:**
*   **Standardized API**: Whether you are using a simple Linear Regression, a Random Forest, or a Support Vector Machine, the code structure (`.fit()`, `.predict()`, `.score()`) is identical.
*   **Batteries Included**: It includes massive toolkits for preprocessing data (scaling, encoding), splitting data, and evaluating models (confusion matrices, ROC curves).

**Real-World Application:** 
A bank wants to predict if a customer will default on a loan. They use Scikit-Learn to train a `RandomForestClassifier` on historical customer data, allowing them to automatically flag high-risk applications in the future.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.datasets import make_classification

# --- Practical Real-World Example: Predicting Loan Defaults ---

# 1. PREPARE: Generate a synthetic dataset representing banking data
# (e.g., Features: Credit Score, Income, Debt. Target: 0 = Paid, 1 = Default)
X, y = make_classification(n_samples=1000, n_features=5, n_classes=2, 
                           weights=[0.85, 0.15], # 85% paid, 15% defaulted
                           random_state=42)

print(f"Dataset Shape: {X.shape} (1000 customers, 5 features)")

# 2. SPLIT: Separate data into Training (80%) and Testing (20%) sets
# We never evaluate a model on the data it was trained on!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. TRAIN: Initialize and fit the Machine Learning model
# We use a Random Forest, a powerful algorithm built from many Decision Trees
model = RandomForestClassifier(n_estimators=50, random_state=42)
print("Training model...")
model.fit(X_train, y_train)

# 4. PREDICT & EVALUATE: Check the accuracy on unseen test data
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("\n--- Model Evaluation ---")
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

# Generate a detailed report showing Precision and Recall
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=["Paid (0)", "Default (1)"]))


# Module 4: Data Serialization & Saving Your Work

After modifying data, analyzing distributions, or training a machine learning model, you need to save it to disk. 

## Using JSON and YAML as Configuration Files
Hardcoding variables directly into your functions is bad practice. If you want to change a parameter (like a file path or a model setting), you have to hunt through your code.

A better way is to use **external configuration files**. JSON and YAML are perfect for this. You store all your settings in a clean, human-readable file, and your Python script simply loads it.

**Advantages:**
*   **Separation of Concerns**: Code is for logic, config files are for settings.
*   **Easy to Change**: Anyone (even non-programmers) can edit a `.yaml` file to change how a script runs.
*   **Reproducibility**: You can share your code along with the exact config file used to get a specific result.

Let's demonstrate this by creating a YAML config file to control a data processing function.


In [ ]:
import json
import yaml

# --- 1. Create the Configuration File ---
# In a real project, this would be a separate file named 'config.yaml'
# We are simulating it here using a multi-line string.
config_yaml_string = """
# Configuration for our data processing pipeline
input_file: 'raw_data.csv'
output_file: 'processed_data.csv'

processing_params:
    mode: 'normalize'  # Can be 'normalize' or 'standardize'
    scale_factor: 100
    
filters:
    - type: 'gaussian'
      kernel_size: 5
    - type: 'median'
      kernel_size: 3
"""

# Let's write this string to a real file so our function can read it
with open("config.yaml", "w") as f:
    f.write(config_yaml_string)

print("--- config.yaml file created ---")
print(config_yaml_string)

# --- 2. Create a Function that USES the Config ---
def process_data_from_config(config_path: str):
    """
    Loads a YAML config file and runs a process based on its settings.
    """
    try:
        with open(config_path, 'r') as f:
            config = yaml.safe_load(f)
            
        print(f"\n--- Running process with config from '{config_path}' ---")
        
        # Access nested parameters like a dictionary
        input_file = config['input_file']
        mode = config['processing_params']['mode']
        filters_to_apply = config['filters']
        
        print(f"Loading data from: {input_file}")
        print(f"Processing mode: {mode}")
        
        for filt in filters_to_apply:
            print(f"Applying filter: {filt['type']} with kernel size {filt['kernel_size']}")
            
    except FileNotFoundError:
        print(f"ERROR: Config file not found at {config_path}")
    except KeyError as e:
        print(f"ERROR: Missing key in config file: {e}")

# --- 3. Run the function ---
# Notice we are NOT passing 'normalize' or 'gaussian' directly.
# The function reads those values from the file.
process_data_from_config("config.yaml")

# --- What about JSON? ---
# The exact same principle applies. Here's the same config in JSON.
config_json_string = """
{
    "input_file": "raw_data.csv",
    "output_file": "processed_data.csv",
    "processing_params": {
        "mode": "normalize",
        "scale_factor": 100
    },
    "filters": [
        {
            "type": "gaussian",
            "kernel_size": 5
        },
        {
            "type": "median",
            "kernel_size": 3
        }
    ]
}
"""
print("\n--- Equivalent JSON Config ---")
print(config_json_string)
